In [ ]:
import os
import sys
import ctypes

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
torch_lib_dir = os.path.join(os.path.dirname(sys.executable), "Lib", "site-packages", "torch", "lib")
if os.path.exists(torch_lib_dir) and hasattr(os, "add_dll_directory"):
    try:
        os.add_dll_directory(torch_lib_dir)
    except Exception:
        pass
for dll_name in ["vcruntime140.dll", "msvcp140.dll", "vcruntime140_1.dll"]:
    try:
        ctypes.CDLL(dll_name)
    except Exception:
        pass

print("System environment initialized successfully.")


In [ ]:
!pip install rdkit
import re
import pickle
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

try:
    from rdkit import Chem
    from rdkit import RDLogger
    RDLogger.DisableLog('rdApp.*')
    RDKIT_AVAILABLE = True
    print("RDKit imported successfully.")
except ImportError:
    RDKIT_AVAILABLE = False
    print("RDKit not installed. Install via `pip install rdkit`.")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

def get_working_device():
    if torch.cuda.is_available():
        try:
            torch.backends.cudnn.enabled = False
            t = torch.zeros(2, 2, device="cuda")
            _ = t + 1.0
            device_name = torch.cuda.get_device_name(0)
            print(f" GPU Accelerator Active: {device_name}")
            return torch.device("cuda")
        except Exception as e:
            print(f" CUDA GPU Error: {e}")
            print("Falling back to CPU mode.")
            return torch.device("cpu")
    print("Using CPU device.")
    return torch.device("cpu")

device = get_working_device()
print(f"Selected compute device: {device}")


In [ ]:
SMILES_REGEX_PATTERN = r"(\[[^\]]+\]|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|\(|\)|\=|\#|\-|\+|\;|\:|\/|\\|\@|\.|\%[0-9]{2}|[0-9])"

class SMILESTokenizer:
    def __init__(self):
        self.pad_token = "<PAD>"
        self.start_token = "<START>"
        self.end_token = "<END>"
        self.unk_token = "<UNK>"
        self.special_tokens = [self.pad_token, self.start_token, self.end_token, self.unk_token]
        self.token_to_idx = {token: idx for idx, token in enumerate(self.special_tokens)}
        self.idx_to_token = {idx: token for idx, token in enumerate(self.special_tokens)}
        
    def tokenize(self, smiles_str):
        return re.findall(SMILES_REGEX_PATTERN, smiles_str)
        
    def fit_on_smiles(self, smiles_list):
        unique_tokens = set()
        for smiles in smiles_list:
            unique_tokens.update(self.tokenize(smiles))
        for token in sorted(list(unique_tokens)):
            if token not in self.token_to_idx:
                idx = len(self.token_to_idx)
                self.token_to_idx[token] = idx
                self.idx_to_token[idx] = token
                
    def encode(self, smiles_str, add_special_tokens=True):
        tokens = self.tokenize(smiles_str)
        indices = [self.token_to_idx.get(tok, self.token_to_idx[self.unk_token]) for tok in tokens]
        if add_special_tokens:
            indices = [self.token_to_idx[self.start_token]] + indices + [self.token_to_idx[self.end_token]]
        return indices

    def decode(self, indices, clean_special=True):
        tokens = []
        for idx in indices:
            token = self.idx_to_token.get(idx, "")
            if clean_special and token in self.special_tokens:
                if token == self.end_token:
                    break
                continue
            tokens.append(token)
        return "".join(tokens)

    @property
    def vocab_size(self):
        return len(self.token_to_idx)

    def save(self, path):
        with open(path, "wb") as f:
            pickle.dump(self.token_to_idx, f)

    @classmethod
    def load(cls, path):
        with open(path, "rb") as f:
            token_to_idx = pickle.load(f)
        tok = cls()
        tok.token_to_idx = token_to_idx
        tok.idx_to_token = {v: k for k, v in token_to_idx.items()}
        return tok


In [ ]:
class SMILESDataset(Dataset):
    def __init__(self, smiles_list, properties_list, tokenizer):
        self.smiles_list = smiles_list
        self.properties_list = properties_list
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.smiles_list)
        
    def __getitem__(self, idx):
        encoded = self.tokenizer.encode(self.smiles_list[idx])
        props = self.properties_list[idx] if self.properties_list is not None else [0.0, 0.0, 0.0]
        return torch.tensor(encoded, dtype=torch.long), torch.tensor(props, dtype=torch.float32)

def smiles_collate_fn(batch):
    sequences, props = zip(*batch)
    padded_seqs = pad_sequence(sequences, batch_first=True, padding_value=0)
    props_tensor = torch.stack(props)
    return padded_seqs, props_tensor


In [ ]:
class ConditionalLSTMVAE(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256, latent_dim=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.num_layers = num_layers
        
        # Shared Embedding Layer
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # Bidirectional Encoder
        self.encoder = nn.LSTM(
            embed_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            bidirectional=True, 
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        # Latent space mean and log-variance projections
        self.fc_mu = nn.Linear(hidden_dim * 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim * 2, latent_dim)
        
        # Initial hidden/cell state projections for decoder
        self.fc_h0 = nn.Linear(latent_dim, hidden_dim * num_layers)
        self.fc_c0 = nn.Linear(latent_dim, hidden_dim * num_layers)
        
        # Decoder LSTM with step-wise latent z injection: Input = [Embedding(x_t) ; z]
        self.decoder = nn.LSTM(
            embed_dim + latent_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        # Vocabulary Output Layer
        self.output_layer = nn.Linear(hidden_dim, vocab_size)
        
        # Auxiliary Property Predictor Head (LogP, QED, SAS)
        self.property_predictor = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )
        
    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu
        
    def encode(self, x):
        x_emb = self.embedding(x)
        _, (h_n, _) = self.encoder(x_emb)
        # Concatenate final hidden states from forward and backward passes
        h_last = torch.cat([h_n[-2], h_n[-1]], dim=-1)
        mu = self.fc_mu(h_last)
        logvar = self.fc_logvar(h_last)
        return mu, logvar
        
    def forward(self, x, dec_input):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        
        # Initialize decoder states
        h0 = self.fc_h0(z).view(self.num_layers, x.size(0), self.hidden_dim).contiguous()
        c0 = self.fc_c0(z).view(self.num_layers, x.size(0), self.hidden_dim).contiguous()
        
        # Step-wise z injection
        dec_emb = self.embedding(dec_input)
        z_seq = z.unsqueeze(1).repeat(1, dec_input.size(1), 1)
        dec_in = torch.cat([dec_emb, z_seq], dim=-1)
        
        decoded, _ = self.decoder(dec_in, (h0, c0))
        logits = self.output_layer(decoded)
        prop_preds = self.property_predictor(z)
        
        return logits, mu, logvar, prop_preds


In [ ]:
def vae_loss_fn(recon_logits, target, mu, logvar, prop_preds, target_props, beta=0.01, gamma=0.1):
    # 1. Reconstruction Loss (CrossEntropy ignoring PAD=0)
    recon_loss = F.cross_entropy(
        recon_logits.view(-1, recon_logits.size(-1)),
        target.view(-1),
        ignore_index=0
    )
    
    # 2. KL Divergence Loss (Sum over latent dim, mean over batch)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=-1).mean()
    
    # 3. Property Prediction Loss (MSE)
    prop_loss = F.mse_loss(prop_preds, target_props) if target_props is not None else torch.tensor(0.0, device=recon_logits.device)
    
    total_loss = recon_loss + beta * kl_loss + gamma * prop_loss
    return total_loss, recon_loss, kl_loss, prop_loss


In [ ]:
# Sampling Pipeline Nucleus Top-p + Temperature 
def generate_smiles_batch(model, tokenizer, num_samples=100, max_len=100, temperature=0.8, top_p=0.9, device="cuda"):
    model.eval()
    generated_smiles = []
    
    start_idx = tokenizer.token_to_idx[tokenizer.start_token]
    end_idx = tokenizer.token_to_idx[tokenizer.end_token]
    
    with torch.no_grad():
        z = torch.randn(num_samples, model.latent_dim, device=device)
        
        h = model.fc_h0(z).view(model.num_layers, num_samples, model.hidden_dim).contiguous()
        c = model.fc_c0(z).view(model.num_layers, num_samples, model.hidden_dim).contiguous()
        
        curr_token = torch.full((num_samples, 1), start_idx, dtype=torch.long, device=device)
        active_mask = torch.ones(num_samples, dtype=torch.bool, device=device)
        generated_indices = [[] for _ in range(num_samples)]
        
        for _ in range(max_len):
            if not active_mask.any():
                break
                
            emb = model.embedding(curr_token) 
            dec_in = torch.cat([emb, z.unsqueeze(1)], dim=-1) 
            
            out, (h, c) = model.decoder(dec_in, (h, c))
            logits = model.output_layer(out[:, -1, :]) / temperature
            
            # Nucleus (Top-p) Filtering
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0
            
            indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
            logits[indices_to_remove] = -float('Inf')
            
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1) # [N, 1]
            
            for i in range(num_samples):
                if active_mask[i]:
                    idx_val = next_token[i].item()
                    if idx_val == end_idx:
                        active_mask[i] = False
                    else:
                        generated_indices[i].append(idx_val)
                        
            curr_token = next_token

    for indices in generated_indices:
        smiles = tokenizer.decode(indices)
        generated_smiles.append(smiles)
        
    return generated_smiles


def evaluate_validity(smiles_list):
    if not RDKIT_AVAILABLE:
        print("RDKit unavailable. Skipping validity check.")
        return 0.0, []
        
    valid_smiles = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(s)
        if mol is not None:
            valid_smiles.append(s)
            
    validity_rate = (len(valid_smiles) / len(smiles_list)) * 100.0
    return validity_rate, valid_smiles


In [ ]:
def train_model(
    csv_path="/kaggle/input/datasets/anushkatayal20/zinc-dataset/250k_rndm_zinc_drugs_clean_3.xls",
    epochs=15,
    batch_size=128,
    learning_rate=0.001,
    latent_dim=128,
    beta_max=0.05
):
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
    else:
        print(f"Dataset file not found at {csv_path}. Using synthetic dummy dataset for demonstration.")
        df = pd.DataFrame({
            "smiles": ["CC(C)(C)c1ccc2occ(CC(=O)Nc3ccccc3F)c2c1", "C[C@@H]1CC(Nc2cncc(-c3nncn3C)c2)C[C@@H](C)C1", "INVALID_SMILES_TEST_###", "CC(C)(C)c1ccc2occ(CC(=O)Nc3ccccc3F)c2c1"] * 250,
            "logP": [5.05, 3.11, 0.00, 5.05] * 250,
            "qed": [0.70, 0.92, 0.00, 0.70] * 250,
            "SAS": [2.08, 3.43, 0.00, 2.08] * 250
        })
        
    original_count = len(df)
    
    #  Remove empty SMILES
    df_step1 = df.dropna(subset=["smiles"]).copy()
    df_step1["smiles"] = df_step1["smiles"].astype(str).str.strip()
    df_step1 = df_step1[df_step1["smiles"] != ""].reset_index(drop=True)
    missing_empty_removed = original_count - len(df_step1)
    
    # RDKit chemical validity check 
    valid_rows = []
    rdkit_invalid_count = 0
    
    for idx in range(len(df_step1)):
        row = df_step1.iloc[idx].to_dict()
        smiles_str = row["smiles"]
        if RDKIT_AVAILABLE:
            mol = Chem.MolFromSmiles(smiles_str)
            if mol is not None:
                # Canonicalize valid SMILES using RDKit
                canon_smiles = Chem.MolToSmiles(mol, canonical=True)
                row["smiles"] = canon_smiles
                valid_rows.append(row)
            else:
                rdkit_invalid_count += 1
        else:
            valid_rows.append(row)
            
    df_valid = pd.DataFrame(valid_rows)
    valid_molecules_count = len(df_valid)
    canonicalized_count = valid_molecules_count
    
    # Remove Duplicate molecules after canonicalization
    count_before_duplicates = len(df_valid)
    df_clean = df_valid.drop_duplicates(subset=["smiles"]).reset_index(drop=True)
    duplicate_removed_count = count_before_duplicates - len(df_clean)
    final_unique_count = len(df_clean)
    
    # Train / Validation Split
    split_idx = int(0.9 * len(df_clean))
    train_df = df_clean.iloc[:split_idx].reset_index(drop=True)
    val_df = df_clean.iloc[split_idx:].reset_index(drop=True)
    train_count = len(train_df)
    val_count = len(val_df)
    
    tokenizer = SMILESTokenizer()
    tokenizer.fit_on_smiles(df_clean["smiles"].tolist())
    vocab_size = tokenizer.vocab_size
  
    print(f"Original molecules: {original_count}")
    print(f"Missing/empty SMILES removed: {missing_empty_removed}")
    print(f"RDKit-invalid SMILES removed: {rdkit_invalid_count}")
    print(f"Valid molecules after RDKit validation: {valid_molecules_count}")
    print(f"Canonicalization completed: {canonicalized_count}")
    print(f"Duplicate molecules removed: {duplicate_removed_count}")
    print(f"Final unique molecules: {final_unique_count}")
    print(f"Training molecules: {train_count}")
    print(f"Validation molecules: {val_count}")
    print(f"Vocabulary Size: {vocab_size}")
    
    train_props = train_df[["logP", "qed", "SAS"]].values if set(["logP", "qed", "SAS"]).issubset(train_df.columns) else None
    val_props = val_df[["logP", "qed", "SAS"]].values if set(["logP", "qed", "SAS"]).issubset(val_df.columns) else None
    
    train_dataset = SMILESDataset(train_df["smiles"].tolist(), train_props, tokenizer)
    val_dataset = SMILESDataset(val_df["smiles"].tolist(), val_props, tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=smiles_collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=smiles_collate_fn)
    
    model = ConditionalLSTMVAE(
        vocab_size=tokenizer.vocab_size,
        embed_dim=256,
        hidden_dim=256,
        latent_dim=latent_dim,
        num_layers=2,
        dropout=0.2
    ).to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    # Modern PyTorch 2.x Mixed Precision Scaler
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
    
    best_val_loss = float("inf")
    
    for epoch in range(epochs):
        model.train()
        total_loss, total_recon, total_kl = 0.0, 0.0, 0.0
        
        # Cyclical beta annealing schedule
        beta = min(beta_max, (epoch + 1) / (epochs / 2) * beta_max)
        
        for x_batch, props_batch in train_loader:
            x_batch = x_batch.to(device)
            props_batch = props_batch.to(device)
            
            # Prepare decoder input: [<START>, x_1, x_2, ..., x_{L-1}]
            dec_in = torch.zeros_like(x_batch)
            dec_in[:, 0] = tokenizer.token_to_idx[tokenizer.start_token]
            dec_in[:, 1:] = x_batch[:, :-1]
            
            optimizer.zero_grad()
            
            # Modern PyTorch 2.x Autocast API
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                logits, mu, logvar, prop_preds = model(x_batch, dec_in)
                loss, recon, kl, prop_l = vae_loss_fn(logits, x_batch, mu, logvar, prop_preds, props_batch, beta=beta)
                
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            
            total_loss += loss.item()
            total_recon += recon.item()
            total_kl += kl.item()
            
        train_loss = total_loss / len(train_loader)
        
        # Validation
        model.eval()
        val_loss_total = 0.0
        with torch.no_grad():
            for x_batch, props_batch in val_loader:
                x_batch = x_batch.to(device)
                props_batch = props_batch.to(device)
                
                dec_in = torch.zeros_like(x_batch)
                dec_in[:, 0] = tokenizer.token_to_idx[tokenizer.start_token]
                dec_in[:, 1:] = x_batch[:, :-1]
                
                with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                    logits, mu, logvar, prop_preds = model(x_batch, dec_in)
                    v_loss, _, _, _ = vae_loss_fn(logits, x_batch, mu, logvar, prop_preds, props_batch, beta=beta)
                val_loss_total += v_loss.item()
                
        val_loss = val_loss_total / len(val_loader)
        
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f} (Recon: {total_recon/len(train_loader):.4f}, KL: {total_kl/len(train_loader):.4f}) | Val Loss: {val_loss:.4f} | Beta: {beta:.4f}")
        
        # Save Best Model Checkpoint & Tokenizer
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_lstm_vae.pt")
            tokenizer.save("tokenizer.pkl")
            
    generated = generate_smiles_batch(model, tokenizer, num_samples=200, device=device)
    validity_rate, valid_smiles = evaluate_validity(generated)
    
    print(f"Chemical Validity Rate: {validity_rate:.2f}%")
    print(f"Sample Generated Valid SMILES:\n{valid_smiles[:5]}")
    
    return model, tokenizer


In [ ]:
model, tokenizer = train_model()

In [ ]:
import os
import torch
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, QED

from improved_lstm_vae_kaggle import (
    ConditionalLSTMVAE, 
    SMILESTokenizer, 
    generate_smiles_batch, 
    device
)

def export_generated_smiles_csv(
    model_path="best_lstm_vae.pt",
    tokenizer_path="tokenizer.pkl",
    output_csv_path="generated_molecules.csv",
    num_samples=500,
    temperature=0.8,
    top_p=0.9
):
    print(f"Loading Checkpoint ({model_path}) & Tokenizer ({tokenizer_path}) ")
    if not os.path.exists(model_path) or not os.path.exists(tokenizer_path):
        raise FileNotFoundError("Model checkpoint or tokenizer file not found. Please run training first!")
        
    tokenizer = SMILESTokenizer.load(tokenizer_path)
    
    model = ConditionalLSTMVAE(
        vocab_size=tokenizer.vocab_size,
        embed_dim=256,
        hidden_dim=256,
        latent_dim=128,
        num_layers=2,
        dropout=0.2
    ).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    print(f" Generating {num_samples} Novel SMILES (Temp={temperature}, Top-p={top_p})")
    raw_smiles_list = generate_smiles_batch(
        model, 
        tokenizer, 
        num_samples=num_samples, 
        temperature=temperature, 
        top_p=top_p, 
        device=device
    )
    
   
    molecule_records = []
    
    for idx, smiles in enumerate(raw_smiles_list):
        mol = Chem.MolFromSmiles(smiles)
        is_valid = mol is not None
        
        if is_valid:
            mw = round(Descriptors.ExactMolWt(mol), 2)
            logp = round(Descriptors.MolLogP(mol), 2)
            hbd = Descriptors.NumHDonors(mol)
            hba = Descriptors.NumHAcceptors(mol)
            qed = round(QED.qed(mol), 4)
            
            # Lipinski Rule of 5 Evaluation
            violations = sum([
                mw > 500,
                logp > 5.0,
                hbd > 5,
                hba > 10
            ])
            lipinski_compliant = violations <= 1
            
            molecule_records.append({
                "Molecule_ID": f"MOL_{idx+1:04d}",
                "SMILES": smiles,
                "Is_Chemically_Valid": True,
                "Molecular_Weight_MW": mw,
                "LogP": logp,
                "HBD": hbd,
                "HBA": hba,
                "QED_Score": qed,
                "Lipinski_Violations": violations,
                "Lipinski_Compliant": lipinski_compliant
            })
        else:
            molecule_records.append({
                "Molecule_ID": f"MOL_{idx+1:04d}",
                "SMILES": smiles,
                "Is_Chemically_Valid": False,
                "Molecular_Weight_MW": None,
                "LogP": None,
                "HBD": None,
                "HBA": None,
                "QED_Score": None,
                "Lipinski_Violations": None,
                "Lipinski_Compliant": False
            })
            
    df_results = pd.DataFrame(molecule_records)
    
    df_results.to_csv(output_csv_path, index=False)
    
    valid_df = df_results[df_results["Is_Chemically_Valid"] == True]
    valid_count = len(valid_df)
    validity_rate = (valid_count / num_samples) * 100.0
    compliant_count = valid_df["Lipinski_Compliant"].sum()
    compliance_rate = (compliant_count / valid_count * 100.0) if valid_count > 0 else 0.0
    mean_qed = valid_df["QED_Score"].mean() if valid_count > 0 else 0.0
    
    print(f"Output File Path:        {os.path.abspath(output_csv_path)}")
    print(f" Total Generated:        {num_samples}")
    print(f" Chemically Valid:        {valid_count} ({validity_rate:.2f}%)")
    print(f" Lipinski Bioavailable:  {compliant_count} ({compliance_rate:.2f}%)")
    print(f" Mean QED Score:         {mean_qed:.4f}")
    
    # Display Preview Table
    print("--- First 5 Generated Molecules Preview ---")
    print(df_results.head())
    
    return df_results

df_generated = export_generated_smiles_csv(num_samples=500)


In [ ]:
import os
import torch
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, QED

def export_generated_smiles_csv(
    model_path="best_lstm_vae.pt",
    tokenizer_path="tokenizer.pkl",
    output_csv_path="generated_molecules.csv",
    num_samples=500,
    temperature=0.8,
    top_p=0.9
):
    print(f" Loading Checkpoint ({model_path}) & Tokenizer ({tokenizer_path}) ---")
    if not os.path.exists(model_path) or not os.path.exists(tokenizer_path):
        raise FileNotFoundError(f"Checkpoint '{model_path}' or '{tokenizer_path}' not found!")
        
    tokenizer = SMILESTokenizer.load(tokenizer_path)
    
    model = ConditionalLSTMVAE(
        vocab_size=tokenizer.vocab_size,
        embed_dim=256,
        hidden_dim=256,
        latent_dim=128,
        num_layers=2,
        dropout=0.2
    ).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    print(f" Generating {num_samples} Novel SMILES (Temp={temperature}, Top-p={top_p}) ")
    raw_smiles_list = generate_smiles_batch(
        model, 
        tokenizer, 
        num_samples=num_samples, 
        temperature=temperature, 
        top_p=top_p, 
        device=device
    )
    
    molecule_records = []
    
    for idx, smiles in enumerate(raw_smiles_list):
        mol = Chem.MolFromSmiles(smiles)
        is_valid = mol is not None
        
        if is_valid:
            mw = round(Descriptors.ExactMolWt(mol), 2)
            logp = round(Descriptors.MolLogP(mol), 2)
            hbd = Descriptors.NumHDonors(mol)
            hba = Descriptors.NumHAcceptors(mol)
            qed = round(QED.qed(mol), 4)
            
            violations = sum([
                mw > 500,
                logp > 5.0,
                hbd > 5,
                hba > 10
            ])
            lipinski_compliant = violations <= 1
            
            molecule_records.append({
                "Molecule_ID": f"MOL_{idx+1:04d}",
                "SMILES": smiles,
                "Is_Chemically_Valid": True,
                "Molecular_Weight_MW": mw,
                "LogP": logp,
                "HBD": hbd,
                "HBA": hba,
                "QED_Score": qed,
                "Lipinski_Violations": violations,
                "Lipinski_Compliant": lipinski_compliant
            })
        else:
            molecule_records.append({
                "Molecule_ID": f"MOL_{idx+1:04d}",
                "SMILES": smiles,
                "Is_Chemically_Valid": False,
                "Molecular_Weight_MW": None,
                "LogP": None,
                "HBD": None,
                "HBA": None,
                "QED_Score": None,
                "Lipinski_Violations": None,
                "Lipinski_Compliant": False
            })
            
    df_results = pd.DataFrame(molecule_records)
    df_results.to_csv(output_csv_path, index=False)
    
    valid_df = df_results[df_results["Is_Chemically_Valid"] == True]
    valid_count = len(valid_df)
    validity_rate = (valid_count / num_samples) * 100.0
    compliant_count = valid_df["Lipinski_Compliant"].sum()
    compliance_rate = (compliant_count / valid_count * 100.0) if valid_count > 0 else 0.0
    mean_qed = valid_df["QED_Score"].mean() if valid_count > 0 else 0.0
    
 
    print(f"Saved CSV Path:          {os.path.abspath(output_csv_path)}")
    print(f"Total Generated:        {num_samples}")
    print(f"Chemically Valid:        {valid_count} ({validity_rate:.2f}%)")
    print(f"Lipinski Bioavailable:  {compliant_count} ({compliance_rate:.2f}%)")
    print(f"Mean QED Score:         {mean_qed:.4f}")
    
    print(df_results.head())
    
    return df_results

df_generated = export_generated_smiles_csv(num_samples=500)


In [ ]:
!pip install rdkit
import os
import sys
import math
import random
import numpy as np
import pandas as pd
import torch

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, QED, AllChem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

SAS_AVAILABLE = False
try:
    from rdkit.Chem import RDConfig
    sas_path = os.path.join(RDConfig.RDContribDir, 'SA_Score')
    if os.path.exists(sas_path) and sas_path not in sys.path:
        sys.path.append(sas_path)
    import sascorer
    SAS_AVAILABLE = True
except Exception:
    SAS_AVAILABLE = False


In [ ]:
def calculate_sa_score_fallback(mol):
    mw = Descriptors.ExactMolWt(mol)
    num_rings = Descriptors.RingCount(mol)
    num_heavy = mol.GetNumHeavyAtoms()
    num_chiral = len(Chem.FindMolChiralCenters(mol, includeUnassigned=True))
    
    score = 1.0 + (mw / 100.0) * 0.5 + (num_rings * 0.4) + (num_chiral * 0.5)
    if num_heavy > 0:
        score += (num_rings / num_heavy) * 2.0
    return min(10.0, max(1.0, round(score, 2)))

def get_sa_score(mol):
    if SAS_AVAILABLE:
        try:
            return round(sascorer.calculateScore(mol), 2)
        except Exception:
            pass
    return calculate_sa_score_fallback(mol)


In [ ]:
import os
import sys
import ctypes
import math
import re
import pickle
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
torch_lib_dir = os.path.join(os.path.dirname(sys.executable), "Lib", "site-packages", "torch", "lib")
if os.path.exists(torch_lib_dir) and hasattr(os, "add_dll_directory"):
    try:
        os.add_dll_directory(torch_lib_dir)
    except Exception:
        pass
for dll_name in ["vcruntime140.dll", "msvcp140.dll", "vcruntime140_1.dll"]:
    try:
        ctypes.CDLL(dll_name)
    except Exception:
        pass

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, QED, AllChem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

SAS_AVAILABLE = False
try:
    from rdkit.Chem import RDConfig
    sas_path = os.path.join(RDConfig.RDContribDir, 'SA_Score')
    if os.path.exists(sas_path) and sas_path not in sys.path:
        sys.path.append(sas_path)
    import sascorer
    SAS_AVAILABLE = True
except Exception:
    SAS_AVAILABLE = False

def get_working_device():
    if torch.cuda.is_available():
        try:
            torch.backends.cudnn.enabled = False
            t = torch.zeros(2, 2, device="cuda")
            _ = t + 1.0
            return torch.device("cuda")
        except Exception:
            return torch.device("cpu")
    return torch.device("cpu")

device = get_working_device()

SMILES_REGEX_PATTERN = r"(\[[^\]]+\]|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|\(|\)|\=|\#|\-|\+|\;|\:|\/|\\|\@|\.|\%[0-9]{2}|[0-9])"

class SMILESTokenizer:
    def __init__(self):
        self.pad_token = "<PAD>"
        self.start_token = "<START>"
        self.end_token = "<END>"
        self.unk_token = "<UNK>"
        self.special_tokens = [self.pad_token, self.start_token, self.end_token, self.unk_token]
        self.token_to_idx = {token: idx for idx, token in enumerate(self.special_tokens)}
        self.idx_to_token = {idx: token for idx, token in enumerate(self.special_tokens)}
        
    def tokenize(self, smiles_str):
        return re.findall(SMILES_REGEX_PATTERN, smiles_str)
        
    def fit_on_smiles(self, smiles_list):
        unique_tokens = set()
        for smiles in smiles_list:
            unique_tokens.update(self.tokenize(smiles))
        for token in sorted(list(unique_tokens)):
            if token not in self.token_to_idx:
                idx = len(self.token_to_idx)
                self.token_to_idx[token] = idx
                self.idx_to_token[idx] = token
                
    def encode(self, smiles_str, add_special_tokens=True):
        tokens = self.tokenize(smiles_str)
        indices = [self.token_to_idx.get(tok, self.token_to_idx[self.unk_token]) for tok in tokens]
        if add_special_tokens:
            indices = [self.token_to_idx[self.start_token]] + indices + [self.token_to_idx[self.end_token]]
        return indices

    def decode(self, indices, clean_special=True):
        tokens = []
        for idx in indices:
            token = self.idx_to_token.get(idx, "")
            if clean_special and token in self.special_tokens:
                if token == self.end_token:
                    break
                continue
            tokens.append(token)
        return "".join(tokens)

    @property
    def vocab_size(self):
        return len(self.token_to_idx)

    def save(self, path):
        with open(path, "wb") as f:
            pickle.dump(self.token_to_idx, f)

    @classmethod
    def load(cls, path):
        with open(path, "rb") as f:
            token_to_idx = pickle.load(f)
        tok = cls()
        tok.token_to_idx = token_to_idx
        tok.idx_to_token = {v: k for k, v in token_to_idx.items()}
        return tok

class ConditionalLSTMVAE(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256, latent_dim=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.encoder = nn.LSTM(
            embed_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            bidirectional=True, 
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        self.fc_mu = nn.Linear(hidden_dim * 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim * 2, latent_dim)
        
        self.fc_h0 = nn.Linear(latent_dim, hidden_dim * num_layers)
        self.fc_c0 = nn.Linear(latent_dim, hidden_dim * num_layers)
        
        self.decoder = nn.LSTM(
            embed_dim + latent_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        self.output_layer = nn.Linear(hidden_dim, vocab_size)
        self.property_predictor = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )
        
    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu
        
    def encode(self, x):
        x_emb = self.embedding(x)
        _, (h_n, _) = self.encoder(x_emb)
        h_last = torch.cat([h_n[-2], h_n[-1]], dim=-1)
        mu = self.fc_mu(h_last)
        logvar = self.fc_logvar(h_last)
        return mu, logvar
        
    def forward(self, x, dec_input):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        
        h0 = self.fc_h0(z).view(self.num_layers, x.size(0), self.hidden_dim).contiguous()
        c0 = self.fc_c0(z).view(self.num_layers, x.size(0), self.hidden_dim).contiguous()
        
        dec_emb = self.embedding(dec_input)
        z_seq = z.unsqueeze(1).repeat(1, dec_input.size(1), 1)
        dec_in = torch.cat([dec_emb, z_seq], dim=-1)
        
        decoded, _ = self.decoder(dec_in, (h0, c0))
        logits = self.output_layer(decoded)
        prop_preds = self.property_predictor(z)
        
        return logits, mu, logvar, prop_preds

def generate_smiles_batch(model, tokenizer, num_samples=100, max_len=100, temperature=0.8, top_p=0.9, device="cuda"):
    model.eval()
    generated_smiles = []
    
    start_idx = tokenizer.token_to_idx[tokenizer.start_token]
    end_idx = tokenizer.token_to_idx[tokenizer.end_token]
    
    with torch.no_grad():
        z = torch.randn(num_samples, model.latent_dim, device=device)
        
        h = model.fc_h0(z).view(model.num_layers, num_samples, model.hidden_dim).contiguous()
        c = model.fc_c0(z).view(model.num_layers, num_samples, model.hidden_dim).contiguous()
        
        curr_token = torch.full((num_samples, 1), start_idx, dtype=torch.long, device=device)
        active_mask = torch.ones(num_samples, dtype=torch.bool, device=device)
        generated_indices = [[] for _ in range(num_samples)]
        
        for _ in range(max_len):
            if not active_mask.any():
                break
                
            emb = model.embedding(curr_token)
            dec_in = torch.cat([emb, z.unsqueeze(1)], dim=-1)
            
            out, (h, c) = model.decoder(dec_in, (h, c))
            logits = model.output_layer(out[:, -1, :]) / temperature
            
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0
            
            indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
            logits[indices_to_remove] = -float('Inf')
            
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            
            for i in range(num_samples):
                if active_mask[i]:
                    idx_val = next_token[i].item()
                    if idx_val == end_idx:
                        active_mask[i] = False
                    else:
                        generated_indices[i].append(idx_val)
                        
            curr_token = next_token

    for indices in generated_indices:
        smiles = tokenizer.decode(indices)
        generated_smiles.append(smiles)
        
    return generated_smiles

def calculate_sa_score_fallback(mol):
    mw = Descriptors.ExactMolWt(mol)
    num_rings = Descriptors.RingCount(mol)
    num_heavy = mol.GetNumHeavyAtoms()
    num_chiral = len(Chem.FindMolChiralCenters(mol, includeUnassigned=True))
    
    score = 1.0 + (mw / 100.0) * 0.5 + (num_rings * 0.4) + (num_chiral * 0.5)
    if num_heavy > 0:
        score += (num_rings / num_heavy) * 2.0
    return min(10.0, max(1.0, round(score, 2)))

def get_sa_score(mol):
    if SAS_AVAILABLE:
        try:
            return round(sascorer.calculateScore(mol), 2)
        except Exception:
            pass
    return calculate_sa_score_fallback(mol)

def evaluate_10k_generation(
    csv_dataset_path="/kaggle/input/datasets/anushkatayal20/zinc-dataset/250k_rndm_zinc_drugs_clean_3.xls",
    model_path="/kaggle/input/models/anushkatayal20/tokenizer-pkl/pytorch/default/1/best_lstm_vae.pt",
    tokenizer_path="/kaggle/input/models/anushkatayal20/tokenizer-pkl/pytorch/default/1/tokenizer.pkl",
    output_csv_path="/kaggle/working/generated_10k_molecules.csv",
    total_samples=5000,
    batch_size=500,
    temperature=0.8,
    top_p=0.9
):

    actual_csv_path = None
    if os.path.exists(csv_dataset_path):
        if os.path.isdir(csv_dataset_path):
            matching_files = [
                os.path.join(csv_dataset_path, f) 
                for f in os.listdir(csv_dataset_path) 
                if f.endswith(('.csv', '.xls', '.txt'))
            ]
            if matching_files:
                actual_csv_path = matching_files[0]
                print(f"Found dataset file in directory: {actual_csv_path}")
        else:
            actual_csv_path = csv_dataset_path

    train_smiles_set = set()
    if actual_csv_path and os.path.exists(actual_csv_path):
        print(f"Loading training dataset from {actual_csv_path} for Novelty evaluation...")
        df_train = pd.read_csv(actual_csv_path)
        if "smiles" in df_train.columns:
            for s in df_train["smiles"].dropna().astype(str).str.strip():
                m = Chem.MolFromSmiles(s)
                if m is not None:
                    train_smiles_set.add(Chem.MolToSmiles(m, canonical=True))
        print(f"Loaded {len(train_smiles_set)} unique reference training molecules.")
    else:
        print("Training dataset file not found.")
    
    if not (os.path.exists(model_path) and os.path.exists(tokenizer_path)):
        raise FileNotFoundError(f"Checkpoint '{model_path}' or '{tokenizer_path}' not found!")
        
    tokenizer = SMILESTokenizer.load(tokenizer_path)
    model = ConditionalLSTMVAE(
        vocab_size=tokenizer.vocab_size,
        embed_dim=256,
        hidden_dim=256,
        latent_dim=128,
        num_layers=2,
        dropout=0.2
    ).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    raw_generated_smiles = []
    
    num_batches = math.ceil(total_samples / batch_size)
    for b in range(num_batches):
        current_batch_size = min(batch_size, total_samples - len(raw_generated_smiles))
        batch_smiles = generate_smiles_batch(
            model,
            tokenizer,
            num_samples=current_batch_size,
            temperature=temperature,
            top_p=top_p,
            device=device
        )
        raw_generated_smiles.extend(batch_smiles)
        print(f"Batch [{b+1}/{num_batches}] completed: Generated {len(raw_generated_smiles)} / {total_samples} molecules.")

    records = []
    valid_mols = []
    valid_canonical_smiles = []
    
    for idx, raw_s in enumerate(raw_generated_smiles):
        mol = Chem.MolFromSmiles(raw_s)
        if mol is not None:
            canon_s = Chem.MolToSmiles(mol, canonical=True)
            mw = round(Descriptors.ExactMolWt(mol), 2)
            logp = round(Descriptors.MolLogP(mol), 2)
            hbd = Descriptors.NumHDonors(mol)
            hba = Descriptors.NumHAcceptors(mol)
            qed_score = round(QED.qed(mol), 4)
            sa_score = get_sa_score(mol)
            
            violations = sum([mw > 500, logp > 5.0, hbd > 5, hba > 10])
            lipinski_ok = violations <= 1
            
            valid_mols.append(mol)
            valid_canonical_smiles.append(canon_s)
            
            records.append({
                "Molecule_ID": f"MOL_{idx+1:05d}",
                "Raw_SMILES": raw_s,
                "Canonical_SMILES": canon_s,
                "Is_Valid": True,
                "MW": mw,
                "LogP": logp,
                "HBD": hbd,
                "HBA": hba,
                "QED": qed_score,
                "SA_Score": sa_score,
                "Lipinski_Violations": violations,
                "Lipinski_Compliant": lipinski_ok
            })
        else:
            records.append({
                "Molecule_ID": f"MOL_{idx+1:05d}",
                "Raw_SMILES": raw_s,
                "Canonical_SMILES": None,
                "Is_Valid": False,
                "MW": None,
                "LogP": None,
                "HBD": None,
                "HBA": None,
                "QED": None,
                "SA_Score": None,
                "Lipinski_Violations": None,
                "Lipinski_Compliant": False
            })

    df_all = pd.DataFrame(records)
    df_all.to_csv(output_csv_path, index=False)
    
    total_gen_count = len(raw_generated_smiles)
    valid_count = len(valid_canonical_smiles)
    validity_pct = (valid_count / total_gen_count) * 100.0 if total_gen_count > 0 else 0.0
    
    unique_smiles_set = set(valid_canonical_smiles)
    unique_count = len(unique_smiles_set)
    uniqueness_pct = (unique_count / valid_count) * 100.0 if valid_count > 0 else 0.0
    
    if len(train_smiles_set) > 0:
        novel_smiles = [s for s in unique_smiles_set if s not in train_smiles_set]
        novelty_pct = (len(novel_smiles) / unique_count) * 100.0 if unique_count > 0 else 0.0
    else:
        novelty_pct = 100.0
        
    fps = []
    for s in list(unique_smiles_set):
        m = Chem.MolFromSmiles(s)
        if m is not None:
            fps.append(AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048))
            
    if len(fps) > 1:
        sample_fps = fps if len(fps) <= 2000 else random.sample(fps, 2000)
        sim_scores = []
        for i in range(len(sample_fps)):
            sims = DataStructs.BulkTanimotoSimilarity(sample_fps[i], sample_fps[i+1:])
            sim_scores.extend(sims)
        mean_sim = np.mean(sim_scores) if len(sim_scores) > 0 else 0.0
        diversity_pct = (1.0 - mean_sim) * 100.0
    else:
        diversity_pct = 0.0

    df_valid = df_all[df_all["Is_Valid"] == True]
    lipinski_count = df_valid["Lipinski_Compliant"].sum()
    lipinski_pct = (lipinski_count / valid_count) * 100.0 if valid_count > 0 else 0.0
    
    mean_qed = df_valid["QED"].mean() if valid_count > 0 else 0.0
    mean_sas = df_valid["SA_Score"].mean() if valid_count > 0 else 0.0


    print(f"Total Molecules Generated:        {total_gen_count:,}")
    print(f"1. Chemical Validity Rate (%):    {validity_pct:.2f}%  ({valid_count:,} valid)")
    print(f"2. Uniqueness Rate (%):          {uniqueness_pct:.2f}%  ({unique_count:,} unique)")
    print(f"3. Novelty Rate (%):             {novelty_pct:.2f}%")
    print(f"4. Internal Diversity (%):       {diversity_pct:.2f}%  (Tanimoto ECFP4 Distance)")
    print(f"5. Lipinski Compliance (%):       {lipinski_pct:.2f}%  ({lipinski_count:,} bioavailable)")
    print(f"6. Mean QED Score (0-1):          {mean_qed:.4f}")
    print(f"7. Mean SA Score (1-10):          {mean_sas:.2f}  (Lower = easier synthesis)")
   
    print(f"Output CSV Saved To:               {os.path.abspath(output_csv_path)}")

    
    return df_all

df_10k_results = evaluate_10k_generation(total_samples=5000, batch_size=500)


In [ ]:
!pip install rdkit
import os
import sys
import ctypes
import math
import re
import pickle
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
torch_lib_dir = os.path.join(os.path.dirname(sys.executable), "Lib", "site-packages", "torch", "lib")
if os.path.exists(torch_lib_dir) and hasattr(os, "add_dll_directory"):
    try:
        os.add_dll_directory(torch_lib_dir)
    except Exception:
        pass
for dll_name in ["vcruntime140.dll", "msvcp140.dll", "vcruntime140_1.dll"]:
    try:
        ctypes.CDLL(dll_name)
    except Exception:
        pass

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, QED, AllChem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

SAS_AVAILABLE = False
try:
    from rdkit.Chem import RDConfig
    sas_path = os.path.join(RDConfig.RDContribDir, 'SA_Score')
    if os.path.exists(sas_path) and sas_path not in sys.path:
        sys.path.append(sas_path)
    import sascorer
    SAS_AVAILABLE = True
except Exception:
    SAS_AVAILABLE = False

def get_working_device():
    if torch.cuda.is_available():
        try:
            torch.backends.cudnn.enabled = False
            t = torch.zeros(2, 2, device="cuda")
            _ = t + 1.0
            return torch.device("cuda")
        except Exception:
            return torch.device("cpu")
    return torch.device("cpu")

device = get_working_device()

SMILES_REGEX_PATTERN = r"(\[[^\]]+\]|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|\(|\)|\=|\#|\-|\+|\;|\:|\/|\\|\@|\.|\%[0-9]{2}|[0-9])"

class SMILESTokenizer:
    def __init__(self):
        self.pad_token = "<PAD>"
        self.start_token = "<START>"
        self.end_token = "<END>"
        self.unk_token = "<UNK>"
        self.special_tokens = [self.pad_token, self.start_token, self.end_token, self.unk_token]
        self.token_to_idx = {token: idx for idx, token in enumerate(self.special_tokens)}
        self.idx_to_token = {idx: token for idx, token in enumerate(self.special_tokens)}
        
    def tokenize(self, smiles_str):
        return re.findall(SMILES_REGEX_PATTERN, smiles_str)
        
    def fit_on_smiles(self, smiles_list):
        unique_tokens = set()
        for smiles in smiles_list:
            unique_tokens.update(self.tokenize(smiles))
        for token in sorted(list(unique_tokens)):
            if token not in self.token_to_idx:
                idx = len(self.token_to_idx)
                self.token_to_idx[token] = idx
                self.idx_to_token[idx] = token
                
    def encode(self, smiles_str, add_special_tokens=True):
        tokens = self.tokenize(smiles_str)
        indices = [self.token_to_idx.get(tok, self.token_to_idx[self.unk_token]) for tok in tokens]
        if add_special_tokens:
            indices = [self.token_to_idx[self.start_token]] + indices + [self.token_to_idx[self.end_token]]
        return indices

    def decode(self, indices, clean_special=True):
        tokens = []
        for idx in indices:
            token = self.idx_to_token.get(idx, "")
            if clean_special and token in self.special_tokens:
                if token == self.end_token:
                    break
                continue
            tokens.append(token)
        return "".join(tokens)

    @property
    def vocab_size(self):
        return len(self.token_to_idx)

    def save(self, path):
        with open(path, "wb") as f:
            pickle.dump(self.token_to_idx, f)

    @classmethod
    def load(cls, path):
        with open(path, "rb") as f:
            token_to_idx = pickle.load(f)
        tok = cls()
        tok.token_to_idx = token_to_idx
        tok.idx_to_token = {v: k for k, v in token_to_idx.items()}
        return tok

class ConditionalLSTMVAE(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256, latent_dim=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.encoder = nn.LSTM(
            embed_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            bidirectional=True, 
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        self.fc_mu = nn.Linear(hidden_dim * 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim * 2, latent_dim)
        
        self.fc_h0 = nn.Linear(latent_dim, hidden_dim * num_layers)
        self.fc_c0 = nn.Linear(latent_dim, hidden_dim * num_layers)
        
        self.decoder = nn.LSTM(
            embed_dim + latent_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        self.output_layer = nn.Linear(hidden_dim, vocab_size)
        self.property_predictor = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )
        
    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu
        
    def encode(self, x):
        x_emb = self.embedding(x)
        _, (h_n, _) = self.encoder(x_emb)
        h_last = torch.cat([h_n[-2], h_n[-1]], dim=-1)
        mu = self.fc_mu(h_last)
        logvar = self.fc_logvar(h_last)
        return mu, logvar
        
    def forward(self, x, dec_input):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        
        h0 = self.fc_h0(z).view(self.num_layers, x.size(0), self.hidden_dim).contiguous()
        c0 = self.fc_c0(z).view(self.num_layers, x.size(0), self.hidden_dim).contiguous()
        
        dec_emb = self.embedding(dec_input)
        z_seq = z.unsqueeze(1).repeat(1, dec_input.size(1), 1)
        dec_in = torch.cat([dec_emb, z_seq], dim=-1)
        
        decoded, _ = self.decoder(dec_in, (h0, c0))
        logits = self.output_layer(decoded)
        prop_preds = self.property_predictor(z)
        
        return logits, mu, logvar, prop_preds

def generate_smiles_batch(model, tokenizer, num_samples=100, max_len=100, temperature=0.8, top_p=0.9, device="cuda"):
    model.eval()
    generated_smiles = []
    
    start_idx = tokenizer.token_to_idx[tokenizer.start_token]
    end_idx = tokenizer.token_to_idx[tokenizer.end_token]
    
    with torch.no_grad():
        z = torch.randn(num_samples, model.latent_dim, device=device)
        
        h = model.fc_h0(z).view(model.num_layers, num_samples, model.hidden_dim).contiguous()
        c = model.fc_c0(z).view(model.num_layers, num_samples, model.hidden_dim).contiguous()
        
        curr_token = torch.full((num_samples, 1), start_idx, dtype=torch.long, device=device)
        active_mask = torch.ones(num_samples, dtype=torch.bool, device=device)
        generated_indices = [[] for _ in range(num_samples)]
        
        for _ in range(max_len):
            if not active_mask.any():
                break
                
            emb = model.embedding(curr_token)
            dec_in = torch.cat([emb, z.unsqueeze(1)], dim=-1)
            
            out, (h, c) = model.decoder(dec_in, (h, c))
            logits = model.output_layer(out[:, -1, :]) / temperature
            
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            
            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0
            
            indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
            logits[indices_to_remove] = -float('Inf')
            
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            
            for i in range(num_samples):
                if active_mask[i]:
                    idx_val = next_token[i].item()
                    if idx_val == end_idx:
                        active_mask[i] = False
                    else:
                        generated_indices[i].append(idx_val)
                        
            curr_token = next_token

    for indices in generated_indices:
        smiles = tokenizer.decode(indices)
        generated_smiles.append(smiles)
        
    return generated_smiles

def calculate_sa_score_fallback(mol):
    mw = Descriptors.ExactMolWt(mol)
    num_rings = Descriptors.RingCount(mol)
    num_heavy = mol.GetNumHeavyAtoms()
    num_chiral = len(Chem.FindMolChiralCenters(mol, includeUnassigned=True))
    
    score = 1.0 + (mw / 100.0) * 0.5 + (num_rings * 0.4) + (num_chiral * 0.5)
    if num_heavy > 0:
        score += (num_rings / num_heavy) * 2.0
    return min(10.0, max(1.0, round(score, 2)))

def get_sa_score(mol):
    if SAS_AVAILABLE:
        try:
            return round(sascorer.calculateScore(mol), 2)
        except Exception:
            pass
    return calculate_sa_score_fallback(mol)

def evaluate_10k_generation(
    csv_dataset_path="/kaggle/input/datasets/anushkatayal20/zinc-dataset/250k_rndm_zinc_drugs_clean_3.xls",
    model_path="/kaggle/input/models/anushkatayal20/tokenizer-pkl/pytorch/default/1/best_lstm_vae.pt",
    tokenizer_path="/kaggle/input/models/anushkatayal20/tokenizer-pkl/pytorch/default/1/tokenizer.pkl",
    output_csv_path="/kaggle/working/generated_10k_molecules.csv",
    total_samples=10000,
    batch_size=500,
    temperature=0.8,
    top_p=0.9
):

    actual_csv_path = None
    if os.path.exists(csv_dataset_path):
        if os.path.isdir(csv_dataset_path):
            matching_files = [
                os.path.join(csv_dataset_path, f) 
                for f in os.listdir(csv_dataset_path) 
                if f.endswith(('.csv', '.xls', '.txt'))
            ]
            if matching_files:
                actual_csv_path = matching_files[0]
                print(f"Found dataset file in directory: {actual_csv_path}")
        else:
            actual_csv_path = csv_dataset_path

    train_smiles_set = set()
    if actual_csv_path and os.path.exists(actual_csv_path):
        print(f"Loading training dataset from {actual_csv_path} for Novelty evaluation...")
        df_train = pd.read_csv(actual_csv_path)
        if "smiles" in df_train.columns:
            for s in df_train["smiles"].dropna().astype(str).str.strip():
                m = Chem.MolFromSmiles(s)
                if m is not None:
                    train_smiles_set.add(Chem.MolToSmiles(m, canonical=True))
        print(f"Loaded {len(train_smiles_set)} unique reference training molecules.")
    else:
        print("Training dataset file not found.")
    
    if not (os.path.exists(model_path) and os.path.exists(tokenizer_path)):
        raise FileNotFoundError(f"Checkpoint '{model_path}' or '{tokenizer_path}' not found!")
        
    tokenizer = SMILESTokenizer.load(tokenizer_path)
    model = ConditionalLSTMVAE(
        vocab_size=tokenizer.vocab_size,
        embed_dim=256,
        hidden_dim=256,
        latent_dim=128,
        num_layers=2,
        dropout=0.2
    ).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    raw_generated_smiles = []
    
    num_batches = math.ceil(total_samples / batch_size)
    for b in range(num_batches):
        current_batch_size = min(batch_size, total_samples - len(raw_generated_smiles))
        batch_smiles = generate_smiles_batch(
            model,
            tokenizer,
            num_samples=current_batch_size,
            temperature=temperature,
            top_p=top_p,
            device=device
        )
        raw_generated_smiles.extend(batch_smiles)
        print(f"Batch [{b+1}/{num_batches}] completed: Generated {len(raw_generated_smiles)} / {total_samples} molecules.")

    records = []
    valid_mols = []
    valid_canonical_smiles = []
    
    for idx, raw_s in enumerate(raw_generated_smiles):
        mol = Chem.MolFromSmiles(raw_s)
        if mol is not None:
            canon_s = Chem.MolToSmiles(mol, canonical=True)
            mw = round(Descriptors.ExactMolWt(mol), 2)
            logp = round(Descriptors.MolLogP(mol), 2)
            hbd = Descriptors.NumHDonors(mol)
            hba = Descriptors.NumHAcceptors(mol)
            qed_score = round(QED.qed(mol), 4)
            sa_score = get_sa_score(mol)
            
            violations = sum([mw > 500, logp > 5.0, hbd > 5, hba > 10])
            lipinski_ok = violations <= 1
            
            valid_mols.append(mol)
            valid_canonical_smiles.append(canon_s)
            
            records.append({
                "Molecule_ID": f"MOL_{idx+1:05d}",
                "Raw_SMILES": raw_s,
                "Canonical_SMILES": canon_s,
                "Is_Valid": True,
                "MW": mw,
                "LogP": logp,
                "HBD": hbd,
                "HBA": hba,
                "QED": qed_score,
                "SA_Score": sa_score,
                "Lipinski_Violations": violations,
                "Lipinski_Compliant": lipinski_ok
            })
        else:
            records.append({
                "Molecule_ID": f"MOL_{idx+1:05d}",
                "Raw_SMILES": raw_s,
                "Canonical_SMILES": None,
                "Is_Valid": False,
                "MW": None,
                "LogP": None,
                "HBD": None,
                "HBA": None,
                "QED": None,
                "SA_Score": None,
                "Lipinski_Violations": None,
                "Lipinski_Compliant": False
            })

    df_all = pd.DataFrame(records)
    df_all.to_csv(output_csv_path, index=False)
    
    total_gen_count = len(raw_generated_smiles)
    valid_count = len(valid_canonical_smiles)
    validity_pct = (valid_count / total_gen_count) * 100.0 if total_gen_count > 0 else 0.0
    
    unique_smiles_set = set(valid_canonical_smiles)
    unique_count = len(unique_smiles_set)
    uniqueness_pct = (unique_count / valid_count) * 100.0 if valid_count > 0 else 0.0
    
    if len(train_smiles_set) > 0:
        novel_smiles = [s for s in unique_smiles_set if s not in train_smiles_set]
        novelty_pct = (len(novel_smiles) / unique_count) * 100.0 if unique_count > 0 else 0.0
    else:
        novelty_pct = 100.0
        
    fps = []
    for s in list(unique_smiles_set):
        m = Chem.MolFromSmiles(s)
        if m is not None:
            fps.append(AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048))
            
    if len(fps) > 1:
        sample_fps = fps if len(fps) <= 2000 else random.sample(fps, 2000)
        sim_scores = []
        for i in range(len(sample_fps)):
            sims = DataStructs.BulkTanimotoSimilarity(sample_fps[i], sample_fps[i+1:])
            sim_scores.extend(sims)
        mean_sim = np.mean(sim_scores) if len(sim_scores) > 0 else 0.0
        diversity_pct = (1.0 - mean_sim) * 100.0
    else:
        diversity_pct = 0.0

    df_valid = df_all[df_all["Is_Valid"] == True]
    lipinski_count = df_valid["Lipinski_Compliant"].sum()
    lipinski_pct = (lipinski_count / valid_count) * 100.0 if valid_count > 0 else 0.0
    
    mean_qed = df_valid["QED"].mean() if valid_count > 0 else 0.0
    mean_sas = df_valid["SA_Score"].mean() if valid_count > 0 else 0.0


    print(f"Total Molecules Generated:        {total_gen_count:,}")
    print(f"1. Chemical Validity Rate (%):    {validity_pct:.2f}%  ({valid_count:,} valid)")
    print(f"2. Uniqueness Rate (%):          {uniqueness_pct:.2f}%  ({unique_count:,} unique)")
    print(f"3. Novelty Rate (%):             {novelty_pct:.2f}%")
    print(f"4. Internal Diversity (%):       {diversity_pct:.2f}%  (Tanimoto ECFP4 Distance)")
    print(f"5. Lipinski Compliance (%):       {lipinski_pct:.2f}%  ({lipinski_count:,} bioavailable)")
    print(f"6. Mean QED Score (0-1):          {mean_qed:.4f}")
    print(f"7. Mean SA Score (1-10):          {mean_sas:.2f}  (Lower = easier synthesis)")
   
    print(f"Output CSV Saved To:               {os.path.abspath(output_csv_path)}")

    
    return df_all

df_10k_results = evaluate_10k_generation(total_samples=10000, batch_size=500)
